# China: Load & Clean  →  `cn_panel_cleaned.csv`

**Источники** (в `data/raw/china/`):

* `active/Wind Software & services.xlsx` — 319 активных IT-компаний, 2014–2025, «tickers в столбцах, metrics в строках».
* `bankrupt/Delisted stocks china.xlsx` — 37 делистнутых компаний, каждая на **отдельном листе** со своим разрезом (dates в колонках, метрики в строках).
* `123.pdf` — ручная классификация 36 делистнутых: **11 real defaults (Target=1)** vs **24 M&A / privatization (Target=0)** + 1 ошибочная (ReneSola, игнорируем).

**Правила научрука (идентично России):**

1. Drop 2025 целиком.
2. `Target==0`: `dropna()` по core-метрикам.
3. `Target==1`: `ffill()` внутри компании, остатки → `fillna(0)`. Никогда не удаляем.
4. `Net Profit` в Wind (`anal_reits_netprofit`) битый — создаём placeholder-столбец `net_profit=0` для всех. Аналогично `net_revenue=0`.

**Целевая схема:** `ticker, company_name, year, total_revenue, ebit, ebitda, total_assets, total_liab, total_equity, current_assets, current_liab, cash, intangibles, cfo, interest_expense, rd_expense, net_profit, net_revenue, target, source_class`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re
import numpy as np
import pandas as pd
from pathlib import Path

CHINA_RAW = Path('../data/raw/china')
WIND = CHINA_RAW / 'active' / 'Wind Software & services.xlsx'
DELIST = CHINA_RAW / 'bankrupt' / 'Delisted stocks china.xlsx'

PROCESSED = Path('../data/processed'); PROCESSED.mkdir(parents=True, exist_ok=True)

print('Wind:   ', WIND, WIND.exists())
print('Delist: ', DELIST, DELIST.exists())

## 1. Классификация делистнутых (из `123.pdf`)

11 компаний = настоящие дефолты (Target=1). 24 компании ушли через M&A/privatization (Target=0). ReneSola (N/A) исключаем.

Ключ матчинга — нормализованное имя (`short_name` из Wind-выгрузки или лист-название).


In [2]:
# Category 1 — True defaults (forced delisting / bankruptcy / non-compliance)
DEFAULT_NAMES = [
    'ChinaCache International Holdings', 'China TechFaith Wireless Comm',
    'China Sunergy Co Ltd', 'Z-Obee Holdings Ltd', 'GEONG International Ltd',
    'China Finance Online Co Ltd', 'LDK Solar Co Ltd', 'LED International Holdings',
    'Link Motion Inc', 'ChinaSing Investment Holdings', 'RCG Holdings Ltd',
]
# Category 0 — Strategic exits (M&A / privatization / voluntary)
STRATEGIC_NAMES = [
    'Actions Semiconductor Co Ltd', 'AutoNavi Holdings Ltd', 'LCT Holdings Ltd',
    'China Mobile Games and Entertainment', 'China Transinfo Technology Corp',
    'Sinotel Technologies', 'iDreamSky Technology Holdings',
    'Elec & Eltek International Co', 'eFuture Information Technology',
    'Sungy Mobile Ltd', 'Gridsum Holding Inc', 'Hollysys Automation Technologies',
    'Hanwha Q Cells Co Ltd', 'iSoftStone Holdings Ltd', 'JA Solar Holdings Co Ltd',
    'KongZhong Corp', 'Sky-Mobi Ltd', 'Montage Technology Group Ltd',
    'NetDimensions (Holdings) Ltd', 'OneConnect Financial Technology',
    'O2Micro International Ltd', 'Qihoo 360 Technology Co Ltd', 'SINA Corp',
    'Semiconductor Manufacturing International Corp',
]
SKIP_NAMES = ['ReneSola Ltd']  # not actually delisted

def _norm(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

DEFAULT_KEYS   = [_norm(x) for x in DEFAULT_NAMES]
STRATEGIC_KEYS = [_norm(x) for x in STRATEGIC_NAMES]
SKIP_KEYS      = [_norm(x) for x in SKIP_NAMES]

def _prefix_match(n, keys, min_common=8):
    """Match if longest common prefix is >= min_common (or full length of shorter)."""
    if not n: return False
    for k in keys:
        if not k: continue
        m = min(len(n), len(k))
        common = 0
        for i in range(m):
            if n[i] == k[i]: common += 1
            else: break
        if common >= min(min_common, m):
            return True
    return False

def classify_delisted(name, sheet_name=None):
    """Return 1=default, 0=strategic, None=skip. Tries both B2 name and sheet name."""
    candidates = [_norm(name)]
    if sheet_name is not None:
        s = re.sub(r'[-\s]*financ.*$', '', str(sheet_name), flags=re.I)
        candidates.append(_norm(s))
    for n in candidates:
        if not n: continue
        if _prefix_match(n, SKIP_KEYS):      return None
        if _prefix_match(n, DEFAULT_KEYS):   return 1
        if _prefix_match(n, STRATEGIC_KEYS): return 0
    return 0  # unknown → treat as strategic (conservative)

print(f'{len(DEFAULT_NAMES)} defaults, {len(STRATEGIC_NAMES)} strategic, {len(SKIP_NAMES)} skip')


11 defaults, 24 strategic, 1 skip


## 2. Wind Active (319 компаний)

Парсим "code row / name row / metric-year rows". Оставляем только нужные метрики.


In [3]:
WIND_METRIC_MAP = {
    'oper_rev':                 'total_revenue',
    'ebit2':                    'ebit',
    'ebitda2':                  'ebitda',
    'tot_assets':               'total_assets',
    'tot_liab':                 'total_liab',
    'tot_equity':               'total_equity',
    'tot_cur_assets':           'current_assets',
    'st_borrow':                'current_liab',         # proxy — Wind не отдаёт explicit CL
    'cash_cash_equ_beg_period': 'cash',
    'intang_assets':            'intangibles',
    'net_cash_flows_oper_act':  'cfo',
    'int_exp':                  'interest_expense',
    'rd_exp':                   'rd_expense',
    # net_profit намеренно не маппим — используем placeholder=0
}

def parse_wind(path):
    raw = pd.read_excel(path, header=None)
    tickers = raw.iloc[0, 2:].tolist()
    names   = raw.iloc[1, 2:].tolist()
    rows = []
    for i in range(raw.shape[0]):
        mtxt = str(raw.iat[i, 0])
        short = raw.iat[i, 1]
        if mtxt in ('代码', 'Name') or pd.isna(short) or short not in WIND_METRIC_MAP:
            continue
        m = re.search(r'\[rptDate\](\d{4})', mtxt)
        if not m: continue
        year = int(m.group(1))
        metric = WIND_METRIC_MAP[short]
        for t, n, v in zip(tickers, names, raw.iloc[i, 2:].tolist()):
            rows.append((t, n, year, metric, v))
    long_df = (pd.DataFrame(rows, columns=['ticker','company_name','year','metric','value'])
                 .dropna(subset=['value'])
                 .drop_duplicates(subset=['ticker','year','metric'], keep='last'))
    wide = long_df.pivot_table(
        index=['ticker','company_name','year'],
        columns='metric', values='value', aggfunc='first'
    ).reset_index()
    wide.columns.name = None
    return wide

wind_panel = parse_wind(WIND)
wind_panel['source_class'] = 'active'
wind_panel['target'] = 0
print(f'Wind: {wind_panel.shape}, tickers={wind_panel.ticker.nunique()}, '
      f'years={sorted(wind_panel.year.unique())}')
wind_panel.head(3)


Wind: (3429, 18), tickers=319, years=[np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,ticker,company_name,year,cash,cfo,current_assets,current_liab,ebit,ebitda,intangibles,interest_expense,rd_expense,total_assets,total_equity,total_liab,total_revenue,source_class,target
0,000004.SZ,GH-TECH,2014,56.469517,-5.534051,264.071607,NaN,3.328633,8.214916,21.181936,NaN,NaN,338.282581,157.384459,180.898122,80.608820,active,0
1,000004.SZ,GH-TECH,2015,39.941107,9.261870,326.529113,10.0,9.075921,14.242866,32.331233,NaN,NaN,398.673507,163.295985,235.377522,120.454422,active,0
2,000004.SZ,GH-TECH,2016,51.516820,147.929759,134.232376,NaN,27.715350,32.250326,24.388896,NaN,NaN,223.716293,178.199802,45.516491,287.670027,active,0


## 3. Delisted per-company sheets (37 листов)

Каждый лист → ticker, name, annual-колонки, label-based извлечение метрик.


In [4]:
DELIST_LABEL_MAP = [
    (('total op rev', 'operating revenue', 'total revenue'),       'total_revenue'),
    (('operating income', 'ebit'),                                  'ebit'),
    (('ebitda',),                                                   'ebitda'),
    (('total assets',),                                             'total_assets'),
    (('total liabilities', 'total liab'),                           'total_liab'),
    (('total equity', "total shareholders' equity",
      "total shareholder's equity", 'total stockholders equity',
      'shareholders equity', "shareholders' equity"),               'total_equity'),
    (('total current assets', 'current assets'),                    'current_assets'),
    (('total current liab', 'current liab'),                        'current_liab'),
    (('cash and cash equivalents', 'cash & cash equivalents',
      'cash and equivalents', 'eop cash balance'),                  'cash'),
    (('intangible assets', 'intangibles'),                          'intangibles'),
    (('short-term borrow', 'short term borrow', 'short-term debt'), 'st_borrow'),
    (('cash flow from operating', 'net cash flow from operating',
      'operating cash flow', 'net cash flows from operating'),      'cfo'),
    (('interest expense',),                                         'interest_expense'),
    (('r&d', 'research and development'),                           'rd_expense'),
]

def _norm_label(s):
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 &]', ' ', str(s).lower())).strip()

def _canonical(label):
    n = _norm_label(label)
    for keys, canon in DELIST_LABEL_MAP:
        if any(k in n for k in keys): return canon
    return None

def parse_delist_sheet(path, sheet):
    df = pd.read_excel(path, sheet_name=sheet, header=None)
    if df.shape[0] < 8 or df.shape[1] < 2: return None
    ticker, name = df.iat[0, 1], df.iat[1, 1]
    if pd.isna(ticker): return None

    date_row = None
    for i in range(4, 12):
        vals = df.iloc[i, 1:].tolist()
        dates, total = 0, 0
        for v in vals:
            if pd.isna(v): continue
            total += 1
            if hasattr(v, 'year'): dates += 1
            elif isinstance(v, (int, float)) and 30000 < v < 80000: dates += 1
            elif isinstance(v, str) and re.search(r'20\d{2}', v): dates += 1
        if total and dates == total: date_row = i; break
    if date_row is None: return None

    period_row = date_row + 1
    raw_dates = df.iloc[date_row, 1:].tolist()
    periods = df.iloc[period_row, 1:].astype(str).str.lower().str.strip().tolist()

    years, keep_cols = [], []
    for j, (d, per) in enumerate(zip(raw_dates, periods)):
        if per not in ('ann.', 'ann', 'annual'): continue
        if pd.isna(d): continue
        yr = None
        if hasattr(d, 'year'): yr = d.year
        elif isinstance(d, (int, float)):
            try: yr = (pd.to_datetime('1899-12-30') + pd.Timedelta(days=int(d))).year
            except Exception: pass
        elif isinstance(d, str):
            m = re.search(r'(20\d{2})', d)
            if m: yr = int(m.group(1))
        if yr:
            years.append(yr); keep_cols.append(j + 1)
    if not keep_cols: return None

    records = {y: {} for y in years}
    for i in range(period_row + 2, df.shape[0]):
        canon = _canonical(df.iat[i, 0])
        if canon is None: continue
        for yr, col in zip(years, keep_cols):
            v = df.iat[i, col]
            if pd.isna(v): continue
            records[yr].setdefault(canon, v)
    out = [{'ticker': ticker, 'company_name': name, 'year': yr, **m}
           for yr, m in records.items() if m]
    return pd.DataFrame(out) if out else None


def load_delisted(path):
    xl = pd.ExcelFile(path)
    frames, log = [], []
    seen_tickers = set()
    for s in xl.sheet_names:
        if re.fullmatch(r'Sheet\d+', s): continue
        parsed = parse_delist_sheet(path, s)
        if parsed is None or parsed.empty:
            log.append((s, 'skip — no annual data')); continue
        tkr = parsed.iat[0, parsed.columns.get_loc('ticker')]
        if tkr in seen_tickers:
            log.append((s, f'skip — duplicate ticker {tkr}')); continue
        seen_tickers.add(tkr)
        b2_name = parsed.iat[0, parsed.columns.get_loc('company_name')]
        cat = classify_delisted(b2_name, sheet_name=s)
        if cat is None:
            log.append((s, 'skip — excluded (ReneSola)')); continue
        parsed['target'] = cat
        parsed['source_class'] = 'default_delisted' if cat == 1 else 'strategic_delisted'
        frames.append(parsed)
        log.append((s, f'ok → target={cat}, name="{b2_name}"'))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(), log

delisted_panel, log = load_delisted(DELIST)
for s, msg in log: print(f'  {s:34s} {msg}')
print(f'\nDelisted combined: {delisted_panel.shape}')
print(f'  target=1: {delisted_panel[delisted_panel.target==1].ticker.nunique()} компаний')
print(f'  target=0: {delisted_panel[delisted_panel.target==0].ticker.nunique()} компаний')


  ACTIONS SEMICONDUCTOR-Financia1    ok → target=0, name="ACTIONS SEMICONDUCTOR"
  AUTONAVI-Financial Summary1        skip — no annual data
  LCT HOLDINGS-Financial Summary1    ok → target=0, name="LCT Holdings (delisting)."
  CHINACACHE-Financial Summary1      ok → target=1, name="CHINACACHE"
  CHINA MOBILE GAMES AND ENTERTA1    ok → target=0, name="CHINA MOBILE GAMES AND ENTERTAINMENT"
  CHINA TECHFAITH WIRELESS COMMU1    ok → target=1, name="CHINA TECHFAITH WIRELESS COMMUNICATION TECHNOLOGY"
  CHINA SUNERGY-Financial Summar1    ok → target=1, name="CHINA SUNERGY"
  CHINA TRANSINFO-Financial Summa    ok → target=0, name="CHINA TRANSINFO"
  SINOTEL TECHNOLOGIES LTD.-Finan    skip — no annual data
  SINOTEL TECHNOLOGIES LTD.-Fina1    ok → target=0, name="SINOTEL TECHNOLOGIES LTD."
  Z-OBEE HOLDINGS LIMITED-Financ1    ok → target=1, name="Z-OBEE HOLDINGS LIMITED"
  IDREAMSKY TECHNOLOGY-Financial     ok → target=0, name="IDREAMSKY TECHNOLOGY"
  ELEC & ELTEK INT CO LTD-Financi    ok → tar

## 4. Merge → единая панель


In [5]:
CORE_COLS = [
    'total_revenue', 'ebit', 'ebitda', 'total_assets', 'total_liab', 'total_equity',
    'current_assets', 'current_liab', 'cash', 'intangibles',
    'cfo', 'interest_expense', 'rd_expense',
]
ID_COLS      = ['ticker', 'company_name', 'year']
SERVICE_COLS = ['target', 'source_class']
ALL_COLS     = ID_COLS + CORE_COLS + ['net_profit', 'net_revenue'] + SERVICE_COLS

# Выравниваем колонки
for frame in (wind_panel, delisted_panel):
    for c in CORE_COLS:
        if c not in frame.columns: frame[c] = np.nan

panel = pd.concat([wind_panel[ID_COLS + CORE_COLS + SERVICE_COLS],
                   delisted_panel[ID_COLS + CORE_COLS + SERVICE_COLS]],
                  ignore_index=True)

# --- placeholder-столбцы (Net Profit / Net Revenue битые/отсутствуют → 0) ---
panel['net_profit']  = 0.0
panel['net_revenue'] = 0.0

panel = panel[ALL_COLS].sort_values(['target','ticker','year'], ascending=[False,True,True]).reset_index(drop=True)

print(f'Combined panel: {panel.shape}')
print(f'Unique tickers: {panel.ticker.nunique()}')
print(f'\nПо source_class:')
print(panel.groupby('source_class')['ticker'].nunique().to_string())
print(f'\nNaN по колонкам (%):')
print((panel[CORE_COLS].isna().mean()*100).round(1).to_string())


Combined panel: (3568, 20)
Unique tickers: 351

По source_class:
source_class
active                319
default_delisted       11
strategic_delisted     21

NaN по колонкам (%):
total_revenue        0.2
ebit                 7.1
ebitda              10.9
total_assets         2.9
total_liab           2.8
total_equity         3.1
current_assets       3.1
current_liab        38.8
cash                 3.2
intangibles         10.1
cfo                  3.2
interest_expense    98.7
rd_expense          30.9


## 5. Supervisor imputation rules

1. **Drop 2025.**
2. `ffill+bfill` внутри каждой компании (лечит точечные пропуски у всех — и активных, и дефолтных).
3. **Target=0:** удаляем строки, где остался NaN в core-backbone (`total_revenue`, `total_assets`, `total_equity`, `current_assets`).
4. **Target=1:** никогда не удаляем; остаточные NaN → `0`.
5. **Actives, остаточные NaN в sparse-столбцах** (`intangibles`, `interest_expense`, `rd_expense`, etc.) → `0` (конвенция "не отражено = 0").


In [6]:
# 1) Drop 2025
before = panel.shape
panel = panel[panel.year != 2025].copy()
print(f'Drop 2025: {before} → {panel.shape}')

# 2) ffill+bfill по компании
panel = panel.sort_values(['ticker','year']).reset_index(drop=True)
panel[CORE_COLS] = (panel.groupby('ticker', group_keys=False)[CORE_COLS]
                        .apply(lambda g: g.ffill().bfill()))
print('После ffill+bfill NaN %:')
print((panel[CORE_COLS].isna().mean()*100).round(1).to_string())


Drop 2025: (3568, 20) → (3464, 20)
После ffill+bfill NaN %:
total_revenue        0.0
ebit                 0.1
ebitda               4.0
total_assets         0.0
total_liab           0.0
total_equity         0.1
current_assets       0.1
current_liab         9.4
cash                 0.1
intangibles          5.2
cfo                  0.1
interest_expense    92.9
rd_expense           0.9


In [7]:
# 3) Target=0: dropna по core-backbone
CORE_BACKBONE = ['total_revenue', 'total_assets', 'total_equity', 'current_assets']
SPARSE_ZERO   = [c for c in CORE_COLS if c not in CORE_BACKBONE]

active = panel[panel.target == 0].copy()
defaults = panel[panel.target == 1].copy()

before_a = len(active)
active = active.dropna(subset=CORE_BACKBONE)
print(f'Target=0: drop rows missing backbone: {before_a:,} → {len(active):,}')

# sparse в активных → 0
active[SPARSE_ZERO] = active[SPARSE_ZERO].fillna(0)

# 4) Target=1: всё что осталось → 0
defaults[CORE_COLS] = defaults[CORE_COLS].fillna(0)

cleaned = pd.concat([active, defaults], ignore_index=True)
cleaned = cleaned.sort_values(['target','ticker','year'], ascending=[False,True,True]).reset_index(drop=True)

print(f'\nAfter cleaning: {cleaned.shape}')
print(f'NaN в core: {cleaned[CORE_COLS].isna().sum().sum()}')


Target=0: drop rows missing backbone: 3,416 → 3,412

After cleaning: (3460, 20)
NaN в core: 0


In [8]:
# --- Контроль классов ---
print('=== Unique companies ===')
print(cleaned.groupby('source_class')['ticker'].nunique().to_string())

print('\n=== Target на уровне строк-лет ===')
print(cleaned['target'].value_counts().rename({0:'Target=0',1:'Target=1'}).to_string())

print('\n=== Unique companies по Target ===')
t1 = cleaned[cleaned.target==1].ticker.nunique()
t0 = cleaned[cleaned.target==0].ticker.nunique()
print(f'  Target=1: {t1}')
print(f'  Target=0: {t0}')
print(f'  imbalance (rows) ≈ 1:{(cleaned.target==0).sum()//max((cleaned.target==1).sum(),1)}')

print('\n=== Годы ===', sorted(cleaned.year.unique()))


=== Unique companies ===
source_class
active                319
default_delisted       11
strategic_delisted     20

=== Target на уровне строк-лет ===
target
Target=0    3412
Target=1      48

=== Unique companies по Target ===
  Target=1: 11
  Target=0: 339
  imbalance (rows) ≈ 1:71

=== Годы === [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [9]:
OUT = PROCESSED / 'cn_panel_cleaned.csv'
cleaned.to_csv(OUT, index=False, encoding='utf-8-sig')
print(f'✅ Saved: {OUT}')
print(f'   size: {OUT.stat().st_size/1024/1024:.1f} MB')
print(f'   shape: {cleaned.shape}')


✅ Saved: ..\data\processed\cn_panel_cleaned.csv
   size: 0.7 MB
   shape: (3460, 20)


## 6. Итог

`data/processed/cn_panel_cleaned.csv` — готов к шагу EDA+моделирования (аналог ноутбука `20_russia_eda_and_models.ipynb`).

* Drop 2025 — сделан.
* 319 active + 24 strategic-delisted = Target 0; 11 real defaults = Target 1.
* `net_profit`, `net_revenue` — placeholder=0 (ждём перезаливку Wind).
* Схема колонок совпадает с ожиданием пайплайна (английские названия, панельный long-format).

⚠️ Ограничения выгрузки Wind (документированы):
* `interest_expense` — фактически 1% заполнен (Wind не отдаёт).
* `current_liab` — proxy через `st_borrow` (~60%); настоящий Current Liabilities отсутствует.
